In [ ]:
import os, subprocess

DATA_DIR = "/kaggle/temp/data"
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(DATA_DIR)
COMP_DIR = "/kaggle/input/competitions/solana-sniper-bot-reverse-engineering"

# 39GB tar (positive + negative class) — background, quiet, resumable
TAR_URL = "http://65.21.203.147:48102/half_year_dataset.tar"
os.system(f"nohup wget -c -q '{TAR_URL}' -O half_year_dataset.tar > tar_download.log 2>&1 &")
print("Tar download running in background at", DATA_DIR)

# wallet activity (~106MB, not mirrored on Kaggle)
WALLET_BASE = "http://154.12.118.112:48114/"
for f in ["5brv79e_activity.parquet", "5brv79e_activity_txs.jsonl.gz", "5brv79e_activity_txs_index.parquet"]:
    if not os.path.exists(f):
        subprocess.run(["wget", "-cq", WALLET_BASE + f], check=True)
print("Done:", os.listdir("."))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "/kaggle/temp/data"
COMP_DIR = "/kaggle/input/competitions/solana-sniper-bot-reverse-engineering"

activity = pd.read_parquet(f"{DATA_DIR}/5brv79e_activity.parquet")
num_cols = ["cost_usd", "buy_cost_usd", "price", "price_usd", "quote_amount",
            "token_amount", "token_total_supply", "gas_native", "gas_usd",
            "dex_native", "dex_usd", "priority_fee", "tip_fee"]
for c in num_cols:
    activity[c] = pd.to_numeric(activity[c], errors="coerce")
wallet_tx_idx = pd.read_parquet(f"{DATA_DIR}/5brv79e_activity_txs_index.parquet")
deploy_idx = pd.read_parquet(f"{COMP_DIR}/bought_deploy_txs_index.parquet")

# attach slot/time to every wallet activity row via tx_hash
activity = activity.merge(wallet_tx_idx[["tx_hash", "blockSlot", "blockTime"]], on="tx_hash", how="left")

buys = activity[activity["event_type"] == "buy"].copy()
sells = activity[activity["event_type"] == "sell"].copy()
burns = activity[activity["event_type"] == "burn"].copy()
print(f"{len(buys)} buys, {len(sells)} sells, {len(burns)} burns, {activity['token_address'].nunique()} unique tokens")

# --- 1. Entry size (USD) ---
first_buys = buys.sort_values("blockSlot").drop_duplicates(subset="token_address", keep="first")
print("Entry size (USD) — mean: %.2f, median: %.2f, std: %.2f" % (
    first_buys["cost_usd"].mean(), first_buys["cost_usd"].median(), first_buys["cost_usd"].std()))
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(first_buys["cost_usd"].clip(upper=first_buys["cost_usd"].quantile(0.99)), bins=60)
ax.set_title("Entry size (USD, clipped p99)"); plt.savefig("entry_size_distribution.png", dpi=150, bbox_inches="tight"); plt.show()

# --- 2. Latency to first buy + zero-block share ---
deploy_lookup = deploy_idx.set_index("token_address")[["blockSlot","blockTime"]].rename(
    columns={"blockSlot":"deploy_slot","blockTime":"deploy_time"})
first_buys = first_buys.join(deploy_lookup, on="token_address")
first_buys["latency_slots"] = first_buys["blockSlot"] - first_buys["deploy_slot"]
first_buys["latency_seconds"] = first_buys["blockTime"] - first_buys["deploy_time"]
zero_block_share = (first_buys["latency_slots"] == 0).mean()
print(f"Zero-block entry share: {zero_block_share:.2%}")
print("Latency (slots) — mean: %.2f, median: %.2f" % (first_buys["latency_slots"].mean(), first_buys["latency_slots"].median()))
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(first_buys["latency_slots"].clip(0,20), bins=21)
ax.set_title("Latency to first buy (slots, clipped 20)"); plt.savefig("latency_histogram.png", dpi=150, bbox_inches="tight"); plt.show()

# --- 3. Hold time / exit structure (burn included) ---
sell_stats = sells.groupby("token_address").agg(n_sell_txs=("tx_hash","nunique"), last_sell_slot=("blockSlot","max"))
burn_tokens = set(burns["token_address"].unique())
hold = first_buys[["token_address","blockSlot"]].rename(columns={"blockSlot":"first_buy_slot"}).merge(
    sell_stats, on="token_address", how="left")
hold["fully_exited"] = hold["last_sell_slot"].notna()
hold["used_burn"] = hold["token_address"].isin(burn_tokens)
hold["hold_slots"] = hold["last_sell_slot"] - hold["first_buy_slot"]
print(f"Tokens exited via sell: {hold['fully_exited'].mean():.2%}")
print(f"Tokens where a burn occurred: {hold['used_burn'].mean():.2%}")
print(f"Multi-tranche exits (>1 sell tx): {(hold['n_sell_txs']>1).mean():.2%} of exited tokens")
print("Hold time (slots) — median: %.1f" % hold["hold_slots"].median())

# --- 4. Hit rate, P&L (USD) ---
buy_cost = buys.groupby("token_address")["cost_usd"].sum().rename("usd_spent")
sell_proceeds = sells.groupby("token_address")["cost_usd"].sum().rename("usd_received")
pnl = pd.concat([buy_cost, sell_proceeds], axis=1).fillna(0)
pnl["realized_pnl_usd"] = pnl["usd_received"] - pnl["usd_spent"]
pnl = pnl[pnl["usd_spent"] > 0]
hit_rate = (pnl["realized_pnl_usd"] > 0).mean()
print(f"Hit rate: {hit_rate:.2%}")
print(f"Avg win: {pnl.loc[pnl['realized_pnl_usd']>0,'realized_pnl_usd'].mean():.2f}, Avg loss: {pnl.loc[pnl['realized_pnl_usd']<=0,'realized_pnl_usd'].mean():.2f}")
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(pnl["realized_pnl_usd"].clip(pnl["realized_pnl_usd"].quantile(0.01), pnl["realized_pnl_usd"].quantile(0.99)), bins=60)
ax.axvline(0, color="black"); ax.set_title("Realized P&L per token (USD, clipped p1-p99)")
plt.savefig("pnl_distribution.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
import pyarrow.parquet as pq, json, gzip
DATA_DIR = "/kaggle/temp/data"
COMP_DIR = "/kaggle/input/competitions/solana-sniper-bot-reverse-engineering"
paths = [f"{COMP_DIR}/bought_deploy_txs_index.parquet", f"{COMP_DIR}/bought_deployers_activity.parquet", f"{DATA_DIR}/not_bought_deploy_txs_index.parquet", f"{DATA_DIR}/not_bought_deployers_activity.parquet"]
pfs = [pq.ParquetFile(p) for p in paths]
[print("\n===", p, "===\nnum_rows:", pf.metadata.num_rows, "\nschema:\n", pf.schema_arrow) for p, pf in zip(paths, pfs)]
samples = [pf.read_row_group(0).to_pandas() for pf in pfs]
[print("\n---sample---", p, "\n", s.head(3).to_string()) for p, s in zip(paths, samples)]

In [ ]:
# Part 2 - Step 1: sample negatives, pull deployer history (t_decision-safe raw data)
import pyarrow.parquet as pq, pandas as pd, numpy as np
SEED = 42
N_NEG_SAMPLE = 80000
DATA_DIR = "/kaggle/temp/data"
COMP_DIR = "/kaggle/input/competitions/solana-sniper-bot-reverse-engineering"
idx_cols = ["tx_hash","blockTime","blockSlot","token_address","tx_signer","creator_address"]
pos_idx = pd.read_parquet(f"{COMP_DIR}/bought_deploy_txs_index.parquet", columns=idx_cols); pos_idx["label"] = 1
neg_idx_all = pd.read_parquet(f"{DATA_DIR}/not_bought_deploy_txs_index.parquet", columns=idx_cols)
neg_idx = neg_idx_all.sample(n=N_NEG_SAMPLE, random_state=SEED).copy(); neg_idx["label"] = 0
deploy_events = pd.concat([pos_idx, neg_idx], ignore_index=True)
print("deploy_events:", deploy_events.shape, "| positives:", int(deploy_events["label"].sum()), "| negatives:", int((deploy_events["label"]==0).sum()))
wallets_pos = set(pos_idx["tx_signer"].dropna().unique())
wallets_neg = set(neg_idx["tx_signer"].dropna().unique())
print("unique deployer wallets - positive:", len(wallets_pos), "| negative sample:", len(wallets_neg))
act_cols = ["wallet","timestamp","event_type","token_address","launchpad_platform","is_open_or_close","cost_usd","gas_usd","token_total_supply","quote_amount"]
pos_hist = pd.read_parquet(f"{COMP_DIR}/bought_deployers_activity.parquet", columns=act_cols)
pos_hist = pos_hist[pos_hist["wallet"].isin(wallets_pos)].copy()
print("pos_hist rows:", pos_hist.shape)
pf_neg_hist = pq.ParquetFile(f"{DATA_DIR}/not_bought_deployers_activity.parquet")
neg_chunks = [b.to_pandas().loc[lambda d: d["wallet"].isin(wallets_neg)] for b in pf_neg_hist.iter_batches(batch_size=300000, columns=act_cols)]
neg_hist = pd.concat(neg_chunks, ignore_index=True)
print("neg_hist rows:", neg_hist.shape)
deployer_hist = pd.concat([pos_hist, neg_hist], ignore_index=True)
print("combined deployer_hist:", deployer_hist.shape)
deployer_hist.to_parquet("/kaggle/temp/data/deployer_hist_sample.parquet")
deploy_events.to_parquet("/kaggle/temp/data/deploy_events_sample.parquet")
print("saved intermediate files")

### t_decision-safe feature engineering (Part 2, Step 2)

All features below are computed strictly using information available **before** the wallet's deploy-time decision (`t_decision`), enforced mechanically rather than by convention.

Key safety mechanism: `pd.merge_asof(..., direction="backward", allow_exact_matches=False)` when attaching each wallet's prior activity history to a deploy event. `allow_exact_matches=False` means a wallet-history row with a timestamp equal to (or after) the deploy timestamp can never be matched — only strictly-prior rows qualify. Combined with `direction="backward"`, this guarantees every cumulative feature (`cum_n_events`, `cum_n_buys`, `cum_cost_usd_sum`, `cum_gas_usd_sum`, `cum_n_distinct_tokens`, `wallet_first_seen`) reflects only what was knowable at or before the moment the wallet made its deploy decision, with no leakage from the deploy event itself or anything after it.

Any wallet with no qualifying prior history (first-ever deploy) has these features filled with 0 / NaT-safe defaults rather than dropped, so cold-start wallets remain in the training set without introducing future information.

In [ ]:
# Part 2 - Step 2: t_decision-safe feature engineering (deployer prior-history + dev-buy signal)
dh = deployer_hist.copy()
dh["cost_usd"] = pd.to_numeric(dh["cost_usd"], errors="coerce")
dh["gas_usd"] = pd.to_numeric(dh["gas_usd"], errors="coerce")
dh["token_total_supply"] = pd.to_numeric(dh["token_total_supply"], errors="coerce")
dh["quote_amount"] = pd.to_numeric(dh["quote_amount"], errors="coerce")
dh = dh.sort_values(["wallet","timestamp"]).reset_index(drop=True)
dh["is_buy"] = (dh["event_type"] == "buy").astype(int)
dh["is_sell"] = (dh["event_type"] == "sell").astype(int)
dh["is_first_occurrence_of_token"] = (~dh.duplicated(subset=["wallet","token_address"], keep="first")).astype(int)
g = dh.groupby("wallet")
dh["cum_n_events"] = g.cumcount() + 1
dh["cum_n_buys"] = g["is_buy"].cumsum()
dh["cum_n_sells"] = g["is_sell"].cumsum()
dh["cum_cost_usd_sum"] = g["cost_usd"].cumsum()
dh["cum_gas_usd_sum"] = g["gas_usd"].cumsum()
dh["cum_n_distinct_tokens"] = g["is_first_occurrence_of_token"].cumsum()
dh["wallet_first_seen"] = g["timestamp"].transform("min")
print("dh engineered:", dh.shape)
de = deploy_events.rename(columns={"tx_signer":"wallet","blockTime":"deploy_time"})[["token_address","wallet","deploy_time","label","blockSlot"]].sort_values("deploy_time").reset_index(drop=True)
dh_lookup = dh[["wallet","timestamp","cum_n_events","cum_n_buys","cum_n_sells","cum_cost_usd_sum","cum_gas_usd_sum","cum_n_distinct_tokens","wallet_first_seen"]].sort_values("timestamp").reset_index(drop=True)
feat = pd.merge_asof(de, dh_lookup, left_on="deploy_time", right_on="timestamp", by="wallet", direction="backward", allow_exact_matches=False)
print("merge_asof (strictly-prior history) done:", feat.shape)
feat["has_prior_activity"] = feat["cum_n_events"].notna().astype(int)
prior_cols = ["cum_n_events","cum_n_buys","cum_n_sells","cum_cost_usd_sum","cum_gas_usd_sum","cum_n_distinct_tokens"]
feat[prior_cols] = feat[prior_cols].fillna(0)
feat["wallet_age_seconds"] = np.where(feat["wallet_first_seen"].notna(), feat["deploy_time"] - feat["wallet_first_seen"], 0)
feat = feat.drop(columns=["timestamp","wallet_first_seen"])
print("prior-history features attached:", feat.shape)
dev_own = dh.merge(de[["wallet","token_address","deploy_time"]], on=["wallet","token_address"], how="inner")
dev_own = dev_own[dev_own["timestamp"] <= dev_own["deploy_time"]]
dev_agg = dev_own.groupby(["wallet","token_address"]).agg(has_dev_buy=("is_buy","max"), dev_buy_cost_usd=("cost_usd","sum"), token_total_supply=("token_total_supply","first"), initial_quote_amount=("quote_amount","first"), launchpad_platform=("launchpad_platform","first")).reset_index()
print("dev_agg rows (deploy-time token info):", dev_agg.shape)
feat = feat.merge(dev_agg, on=["wallet","token_address"], how="left")
feat["has_dev_buy"] = feat["has_dev_buy"].fillna(0).astype(int)
feat["dev_buy_cost_usd"] = feat["dev_buy_cost_usd"].fillna(0)
print("final feature table:", feat.shape)
feat.to_parquet("/kaggle/temp/data/part2_features.parquet")
print(feat["label"].value_counts())
print(feat.isna().sum())
feat.head(10)

In [ ]:
# Part 2 - Step 2a: reload intermediates + shrink memory footprint (kernel restarted from OOM, retry with lighter dtypes)
import pandas as pd, numpy as np, gc
DATA_DIR = "/kaggle/temp/data"
deployer_hist = pd.read_parquet(f"{DATA_DIR}/deployer_hist_sample.parquet")
deploy_events = pd.read_parquet(f"{DATA_DIR}/deploy_events_sample.parquet")
print("loaded raw:", deployer_hist.shape, deploy_events.shape)
print("raw memory MB:", round(deployer_hist.memory_usage(deep=True).sum()/1e6, 1))
deployer_hist["event_type"] = deployer_hist["event_type"].astype("category")
deployer_hist["launchpad_platform"] = deployer_hist["launchpad_platform"].astype("category")
deployer_hist["cost_usd"] = pd.to_numeric(deployer_hist["cost_usd"], errors="coerce").astype("float32")
deployer_hist["gas_usd"] = pd.to_numeric(deployer_hist["gas_usd"], errors="coerce").astype("float32")
deployer_hist["token_total_supply"] = pd.to_numeric(deployer_hist["token_total_supply"], errors="coerce").astype("float32")
deployer_hist["quote_amount"] = pd.to_numeric(deployer_hist["quote_amount"], errors="coerce").astype("float32")
deployer_hist["timestamp"] = deployer_hist["timestamp"].astype("int64")
deployer_hist = deployer_hist.drop(columns=["is_open_or_close"])
gc.collect()
print("optimized memory MB:", round(deployer_hist.memory_usage(deep=True).sum()/1e6, 1))
deploy_events["blockTime"] = deploy_events["blockTime"].astype("int64")
deploy_events["label"] = deploy_events["label"].astype("int8")
print(deployer_hist.dtypes)
print(deploy_events.dtypes)

In [ ]:
# Part 2 - Step 2b: t_decision-safe feature engineering (no full-frame .copy(), memory-safe)
dh = deployer_hist
dh = dh.sort_values(["wallet","timestamp"], ignore_index=True)
gc.collect()
dh["is_buy"] = (dh["event_type"] == "buy").astype("int8")
dh["is_first_occurrence_of_token"] = (~dh.duplicated(subset=["wallet","token_address"], keep="first")).astype("int8")
g = dh.groupby("wallet", sort=False)
dh["cum_n_events"] = (g.cumcount() + 1).astype("int32")
dh["cum_n_buys"] = g["is_buy"].cumsum().astype("int32")
dh["cum_cost_usd_sum"] = g["cost_usd"].cumsum().astype("float32")
dh["cum_gas_usd_sum"] = g["gas_usd"].cumsum().astype("float32")
dh["cum_n_distinct_tokens"] = g["is_first_occurrence_of_token"].cumsum().astype("int32")
dh["wallet_first_seen"] = g["timestamp"].transform("min")
del g
gc.collect()
print("dh engineered:", dh.shape, "| memory MB:", round(dh.memory_usage(deep=True).sum()/1e6, 1))

In [ ]:
# Part 2 - Step 2c: merge_asof (strictly-prior deployer history) + dev-buy / deploy-time token info
de = deploy_events.rename(columns={"tx_signer":"wallet","blockTime":"deploy_time"})[["token_address","wallet","deploy_time","label","blockSlot"]].sort_values("deploy_time", ignore_index=True)
dh_lookup = dh[["wallet","timestamp","cum_n_events","cum_n_buys","cum_cost_usd_sum","cum_gas_usd_sum","cum_n_distinct_tokens","wallet_first_seen"]].sort_values("timestamp", ignore_index=True)
feat = pd.merge_asof(de, dh_lookup, left_on="deploy_time", right_on="timestamp", by="wallet", direction="backward", allow_exact_matches=False)
del dh_lookup
gc.collect()
print("merge_asof done:", feat.shape)
feat["has_prior_activity"] = feat["cum_n_events"].notna().astype("int8")
prior_cols = ["cum_n_events","cum_n_buys","cum_cost_usd_sum","cum_gas_usd_sum","cum_n_distinct_tokens"]
feat[prior_cols] = feat[prior_cols].fillna(0)
feat["wallet_age_seconds"] = np.where(feat["wallet_first_seen"].notna(), feat["deploy_time"] - feat["wallet_first_seen"], 0)
feat = feat.drop(columns=["timestamp","wallet_first_seen"])
print("prior-history features attached:", feat.shape)
dev_own = dh[dh["token_address"].isin(set(de["token_address"]))][["wallet","token_address","timestamp","is_buy","cost_usd","token_total_supply","quote_amount","launchpad_platform"]].merge(de[["wallet","token_address","deploy_time"]], on=["wallet","token_address"], how="inner")
dev_own = dev_own[dev_own["timestamp"] <= dev_own["deploy_time"]]
dev_agg = dev_own.groupby(["wallet","token_address"]).agg(has_dev_buy=("is_buy","max"), dev_buy_cost_usd=("cost_usd","sum"), token_total_supply=("token_total_supply","first"), initial_quote_amount=("quote_amount","first"), launchpad_platform=("launchpad_platform","first")).reset_index()
print("dev_agg rows:", dev_agg.shape)
feat = feat.merge(dev_agg, on=["wallet","token_address"], how="left")
feat["has_dev_buy"] = feat["has_dev_buy"].fillna(0).astype("int8")
feat["dev_buy_cost_usd"] = feat["dev_buy_cost_usd"].fillna(0)
print("final feature table:", feat.shape)
feat.to_parquet("/kaggle/temp/data/part2_features.parquet")
print(feat["label"].value_counts())
print(feat.isna().sum())

In [ ]:
# Part 2 - Step 3: interpretable classifier (LightGBM), time-based split, precision/recall/F1/PR-AUC
import lightgbm as lgb
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, classification_report
feature_cols = ["cum_n_events","cum_n_buys","cum_cost_usd_sum","cum_gas_usd_sum","cum_n_distinct_tokens","has_prior_activity","wallet_age_seconds","has_dev_buy","dev_buy_cost_usd","token_total_supply","initial_quote_amount","launchpad_platform"]
model_df = feat.sort_values("deploy_time", ignore_index=True).copy()
model_df["launchpad_platform"] = model_df["launchpad_platform"].astype("category")
cutoff = int(len(model_df) * 0.8)
train_df = model_df.iloc[:cutoff]
test_df = model_df.iloc[cutoff:]
print("train:", train_df.shape, "test:", test_df.shape)
print("train label rate:", round(train_df["label"].mean(),4), "| test label rate:", round(test_df["label"].mean(),4))
X_train, y_train = train_df[feature_cols], train_df["label"]
X_test, y_test = test_df[feature_cols], test_df["label"]
clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train, categorical_feature=["launchpad_platform"])
proba = clf.predict_proba(X_test)[:,1]
pr_auc = average_precision_score(y_test, proba)
preds = (proba >= 0.5).astype(int)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average="binary", pos_label=1)
print(f"PR-AUC (bought class): {pr_auc:.4f}")
print(f"At threshold 0.5 - Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
print(classification_report(y_test, preds, target_names=["not_bought","bought"]))

In [ ]:
# Part 2 - Step 4: SHAP interpretability - top-10 feature importances
import shap
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values
mean_abs_shap = pd.Series(np.abs(sv).mean(axis=0), index=feature_cols).sort_values(ascending=False)
print("Top-10 features by mean |SHAP| (impact on P(bought)):")
print(mean_abs_shap.head(10))
mean_abs_shap.to_csv("/kaggle/working/part2_shap_importance.csv")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,5))
mean_abs_shap.head(10).sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top-10 feature importances (mean |SHAP value|)")
ax.set_xlabel("mean |SHAP value|")
plt.tight_layout()
plt.savefig("part2_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Part 3 - Step 1: reload bot's own wallet activity, recompute per-token realized P&L
activity = pd.read_parquet("/kaggle/temp/data/5brv79e_activity.parquet")
num_cols = ["cost_usd","buy_cost_usd","price","price_usd","quote_amount","token_amount","token_total_supply","gas_native","gas_usd","dex_native","dex_usd","priority_fee","tip_fee"]
activity[num_cols] = activity[num_cols].apply(lambda s: pd.to_numeric(s, errors="coerce"))
buys = activity[activity["event_type"]=="buy"]
sells = activity[activity["event_type"]=="sell"]
buy_cost = buys.groupby("token_address")["cost_usd"].sum().rename("usd_spent")
sell_proceeds = sells.groupby("token_address")["cost_usd"].sum().rename("usd_received")
pnl = pd.concat([buy_cost, sell_proceeds], axis=1).fillna(0)
pnl["realized_pnl_usd"] = pnl["usd_received"] - pnl["usd_spent"]
pnl = pnl[pnl["usd_spent"] > 0].reset_index()
first_buy = buys.sort_values("timestamp").drop_duplicates(subset="token_address", keep="first")[["token_address","timestamp"]].rename(columns={"timestamp":"first_buy_time"})
pnl = pnl.merge(first_buy, on="token_address", how="left")
print("pnl table:", pnl.shape)
test_results = test_df[["token_address","label","deploy_time"]].copy()
test_results["proba"] = proba
test_results["pred"] = (test_results["proba"] >= 0.5).astype(int)
test_results = test_results.merge(pnl[["token_address","realized_pnl_usd","first_buy_time"]], on="token_address", how="left")
print("test_results:", test_results.shape)
print(test_results.head(10))

In [ ]:
# Part 3 - Step 2: backtest - bot's actual performance vs replica strategy (true positives) + equity curve
bot_actual = test_results[test_results["label"]==1].copy()
replica_tp = test_results[(test_results["label"]==1) & (test_results["pred"]==1)].copy()
false_positives = test_results[(test_results["label"]==0) & (test_results["pred"]==1)].copy()
print("BOT ACTUAL: n=", len(bot_actual), "| hit_rate=", round((bot_actual["realized_pnl_usd"]>0).mean(),4), "| avg_win=", round(bot_actual.loc[bot_actual["realized_pnl_usd"]>0,"realized_pnl_usd"].mean(),2), "| avg_loss=", round(bot_actual.loc[bot_actual["realized_pnl_usd"]<=0,"realized_pnl_usd"].mean(),2), "| total_pnl=", round(bot_actual["realized_pnl_usd"].sum(),2))
print("REPLICA TRUE POSITIVES: n=", len(replica_tp), "| hit_rate=", round((replica_tp["realized_pnl_usd"]>0).mean(),4), "| avg_win=", round(replica_tp.loc[replica_tp["realized_pnl_usd"]>0,"realized_pnl_usd"].mean(),2), "| avg_loss=", round(replica_tp.loc[replica_tp["realized_pnl_usd"]<=0,"realized_pnl_usd"].mean(),2), "| total_pnl=", round(replica_tp["realized_pnl_usd"].sum(),2))
print("Coverage (recall) =", round(len(replica_tp)/len(bot_actual),4), "| False positives (unknown P&L) =", len(false_positives), "of", len(test_results[test_results['pred']==1]), "replica buys")
bot_curve = bot_actual.sort_values("first_buy_time")["realized_pnl_usd"].cumsum().reset_index(drop=True)
replica_curve = replica_tp.sort_values("first_buy_time")["realized_pnl_usd"].cumsum().reset_index(drop=True)
bot_dd = (bot_curve - bot_curve.cummax()).min()
replica_dd = (replica_curve - replica_curve.cummax()).min()
print("Bot max drawdown (USD):", round(bot_dd,2), "| Replica max drawdown (USD):", round(replica_dd,2))
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(range(len(bot_curve)), bot_curve.values, label="Bot actual (all test-period buys)")
ax.plot(range(len(replica_curve)), replica_curve.values, label="Replica strategy (true positives)")
ax.set_title("Equity curve - cumulative realized P&L (USD)")
ax.set_xlabel("trade sequence")
ax.set_ylabel("cumulative P&L (USD)")
ax.legend()
plt.savefig("/kaggle/working/part3_equity_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Part 3 - Step 3: entry-delay sensitivity (proxy: 1st-buy vs 2nd-buy price on multi-tranche tokens, since no per-slot price feed is available)
buys_sorted = buys.sort_values(["token_address","timestamp"]).copy()
buys_sorted["buy_seq"] = buys_sorted.groupby("token_address").cumcount() + 1
first_two = buys_sorted[buys_sorted["buy_seq"]<=2]
price_pivot = first_two.pivot_table(index="token_address", columns="buy_seq", values="price_usd")
price_pivot.columns = ["buy1_price","buy2_price"]
price_pivot = price_pivot.dropna()
price_pivot["pct_change"] = (price_pivot["buy2_price"] - price_pivot["buy1_price"]) / price_pivot["buy1_price"] * 100
tp_tokens = set(replica_tp["token_address"])
delay_sensitivity = price_pivot[price_pivot.index.isin(tp_tokens)]
print("true-positive tokens with a 2nd buy tx (delay proxy):", len(delay_sensitivity), "of", len(replica_tp))
print("median price change 1st->2nd buy:", round(delay_sensitivity["pct_change"].median(),2), "% | mean:", round(delay_sensitivity["pct_change"].mean(),2), "%")
print("share where a delayed entry costs more (price rose):", round((delay_sensitivity["pct_change"]>0).mean()*100,2), "%")
sim = pnl[["token_address","usd_spent","usd_received","realized_pnl_usd"]].merge(delay_sensitivity[["buy1_price","buy2_price"]], left_on="token_address", right_index=True, how="inner")
sim["delayed_usd_spent"] = sim["usd_spent"] * (sim["buy2_price"]/sim["buy1_price"])
sim["delayed_pnl_usd"] = sim["usd_received"] - sim["delayed_usd_spent"]
print("baseline (actual-slot) hit rate:", round((sim["realized_pnl_usd"]>0).mean(),4), "| delayed-entry hit rate:", round((sim["delayed_pnl_usd"]>0).mean(),4))
print("baseline total P&L:", round(sim["realized_pnl_usd"].sum(),2), "| delayed-entry total P&L:", round(sim["delayed_pnl_usd"].sum(),2))